In [1]:
# Imports
from chessdotcom import *
import pprint
import requests

import urllib.request
import json

import numpy as np
import pandas as pd
import pickle
import json
import time

from datetime import datetime

In [2]:
## Configure Headers, as the Project uses requests package to interact with the API. Headers and paxis can be set
## through the Client Object.
import asyncio
from chessdotcom.aio import get_player_profile, Client
# Set aio=True to enable asynchronous operations
Client.aio = True

Client.request_config["headers"]["User-Agent"] = (
 "Machine learning for chess match outcome Prediction, BSc in Computer Science Dissertation, University of Lincoln"
 "Contact me at conorjackvincent@live.co.uk"
)

Client.rate_limit_handler.tries = 2
Client.rate_limit_handler.tts = 4

In [3]:
## Load the player_profiles_orginal.pkl file, so that we can use this pandas dataframes usernames to collect further statistical data in terms of game history.
with open('player_profiles_original.pkl', 'rb') as file:
    player_profiles = pickle.load(file)

In [4]:
player_profiles.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10204 entries, 0 to 354
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   name          8115 non-null   object
 1   username      10204 non-null  object
 2   title         10204 non-null  object
 3   followers     10204 non-null  int64 
 4   country_code  10204 non-null  object
 5   country       10204 non-null  object
 6   status        10204 non-null  object
 7   is_streamer   10204 non-null  bool  
 8   verified      10204 non-null  bool  
 9   league        8474 non-null   object
dtypes: bool(2), int64(1), object(7)
memory usage: 737.4+ KB


In [5]:
# Create a list containing all the usernames in the username column of player_profiles pandas df.
usernames = player_profiles['username'].tolist()

In [6]:
cors = [get_player_game_archives(username) for username in usernames]

In [7]:
print(len(cors))

10204


In [8]:
print(cors[0])

<coroutine object Client._do_async_get_request at 0x000001E979340890>


In [9]:
# https://stackoverflow.com/questions/48483348/how-to-limit-concurrency-with-python-asyncio/61478547#61478547
# https://stackoverflow.com/questions/47675410/python-asyncio-aiohttp-valueerror-too-many-file-descriptors-in-select-on-win

async def gather_with_concurrency(n, *coros):
    semaphore = asyncio.Semaphore(n)

    async def sem_coro(coro):
        async with semaphore:
            return await coro
    return await asyncio.gather(*(sem_coro(c) for c in coros))

In [10]:
responses = await gather_with_concurrency(10, *cors)

ChessDotComError: <class 'chessdotcom.types.ChessDotComError'>(status_code=429, text=)